# 06 — XGBoost Risk Model

**Question:** can we predict supply-chain risk (y ∈ [0,1]) from country + product features well enough to be useful in the UI?

**Approach:** XGBoost regressor + Hyperopt search + 5-fold CV.  Target RMSE ≤ 0.15.

## Setup

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, joblib, json
import xgboost as xgb
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

SEED = 42; np.random.seed(SEED)
FEAT = pd.read_parquet('../data/processed/risk_features.parquet')
ART = Path('../ml/artifacts'); ART.mkdir(parents=True, exist_ok=True)
FIG = Path('../docs/figures'); FIG.mkdir(parents=True, exist_ok=True)
FEAT.head()

## Split

In [ ]:
num_cols = ['distance_km','tariff_pct','trade_value_log','country_risk','source_concentration','historic_disruption_count','free_zone_flag']
X = FEAT[[c for c in num_cols if c in FEAT.columns]].fillna(FEAT[num_cols[0]].median() if num_cols[0] in FEAT else 0)
y = FEAT['y_risk_score']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED)

## Hyperopt (small budget for the notebook run)

In [ ]:
from hyperopt import fmin, tpe, hp, Trials

space = {
    'max_depth':      hp.choice('max_depth', [3,4,5,6,8]),
    'learning_rate':  hp.loguniform('learning_rate', np.log(1e-3), np.log(3e-1)),
    'n_estimators':   hp.choice('n_estimators', [200,400,800]),
    'subsample':      hp.uniform('subsample', 0.6, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.6, 1.0),
}

def objective(p):
    m = xgb.XGBRegressor(**p, random_state=SEED, n_jobs=-1, verbosity=0)
    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    losses = []
    for tr, va in kf.split(X_tr):
        m.fit(X_tr.iloc[tr], y_tr.iloc[tr])
        losses.append(np.sqrt(mean_squared_error(y_tr.iloc[va], m.predict(X_tr.iloc[va]))))
    return np.mean(losses)

trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=30, trials=trials, rstate=np.random.default_rng(SEED))
print(best)

## Refit on full train, eval on test

In [ ]:
params = {
    'max_depth': [3,4,5,6,8][best['max_depth']],
    'learning_rate': best['learning_rate'],
    'n_estimators': [200,400,800][best['n_estimators']],
    'subsample': best['subsample'], 'colsample_bytree': best['colsample_bytree'],
}
model = xgb.XGBRegressor(**params, random_state=SEED, n_jobs=-1)
model.fit(X_tr, y_tr)
p_te = model.predict(X_te)
rmse = np.sqrt(mean_squared_error(y_te, p_te)); r2 = r2_score(y_te, p_te)
print(f'TEST rmse={rmse:.4f}  r2={r2:.3f}')

## Calibration curve

In [ ]:
bins = pd.cut(p_te, 10)
cal = pd.DataFrame({'pred':p_te,'true':y_te.values,'bin':bins}).groupby('bin').mean()
plt.figure(figsize=(4,4))
plt.plot([0,1],[0,1],'--',c='gray')
plt.scatter(cal.pred, cal['true'], c='#7c5cff')
plt.xlabel('predicted'); plt.ylabel('actual'); plt.title('Calibration')
plt.tight_layout(); plt.savefig(FIG/'06_calibration.png', dpi=140)

## Save artifact

In [ ]:
joblib.dump({'model': model, 'features': list(X.columns)}, ART/'risk_xgb.pkl')
(ART/'risk_xgb_metrics.json').write_text(json.dumps({'test_rmse': float(rmse), 'test_r2': float(r2), 'features': list(X.columns), 'best_params': params}, indent=2))